In [ ]:
!pip install -q python-docx pypdf pandas openpyxl dateparser spacy

!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.5/300.5 kB 20.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import os
import pandas as pd
from dateparser.search import search_dates
import spacy

from google.colab import files
from docx import Document
from pypdf import PdfReader

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
uploaded = files.upload()

Saving Sample_Train_Ticket.pdf to Sample_Train_Ticket.pdf


In [ ]:
def extract_text(file_path):

    ext = os.path.splitext(file_path)[1].lower()

    text = ""

    if ext == ".pdf":

        reader = PdfReader(file_path)

        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text() + "\n"

    elif ext == ".docx":

        doc = Document(file_path)

        text = "\n".join([p.text for p in doc.paragraphs])

    elif ext == ".csv":

        df = pd.read_csv(file_path)

        text = df.to_string(index=False)

    elif ext in [".xlsx", ".xls"]:

        excel = pd.ExcelFile(file_path)

        for sheet in excel.sheet_names:

            df = pd.read_excel(file_path, sheet_name=sheet)

            text += df.to_string(index=False)

    elif ext == ".txt":

        with open(file_path, "r", encoding="utf-8") as f:

            text = f.read()

    return text

In [ ]:
deadline_keywords = [
    "boarding",
    "gate closes",
    "reporting",
    "last date",
    "submit",
    "submission",
    "due",
    "closing",
    "expiry",
    "expiration",
    "registration",
    "exam",
    "interview",
    "meeting",
    "renewal",
    "deadline" # Generic 'deadline' is now last
]

In [ ]:
def extract_deadlines(text, source):
    results = []
    doc = nlp(text)  # Use spaCy for sentence processing

    for sent in doc.sents:
        sentence_text = sent.text.strip()
        # Use strict parsing to reduce false positives for dates
        dates_found = search_dates(sentence_text, settings={'STRICT_PARSING': True})

        if dates_found:
            for matched_string, datetime_object in dates_found:
                # Find the character span of the matched date within the sentence
                date_start = sentence_text.find(matched_string)
                date_end = date_start + len(matched_string)

                # Define a context window around the date string (e.g., +/- 50 characters)
                window_size = 50
                context_start = max(0, date_start - window_size)
                context_end = min(len(sentence_text), date_end + window_size)
                context_text = sentence_text[context_start:context_end].lower()

                matching_keyword = None
                for keyword in deadline_keywords:
                    if keyword in context_text:
                        matching_keyword = keyword
                        break

                if matching_keyword:
                    event_label_prefix = ""
                    # Construct Event using the matched keyword and the original date string
                    if matching_keyword.lower() == 'deadline':
                        event_label_prefix = "Deadline"
                    else:
                        event_label_prefix = f"{matching_keyword.title()} Deadline"

                    event_description = f"{event_label_prefix}: {matched_string}" # Minor comment change to force update

                    results.append({
                        "Document": source,
                        "Deadline": datetime_object, # Store datetime object directly for proper sorting
                        "Event": event_description,
                        "MatchedString": matched_string # Store the exact matched string for precision comparison
                    })
    return results

In [ ]:
all_deadlines = []

for file_name in uploaded.keys():

    print("Processing:", file_name)

    text = extract_text(file_name)

    deadlines = extract_deadlines(text, file_name)

    all_deadlines.extend(deadlines)

Processing: Sample_Train_Ticket.pdf


In [ ]:
import pandas as pd

df = pd.DataFrame(all_deadlines)

# Check if the DataFrame is empty before attempting to sort
if not df.empty:
    df = df.drop_duplicates()
    df['Deadline'] = pd.to_datetime(df['Deadline']) # Ensure 'Deadline' is datetime

    # Helper columns for consolidation logic
    df['DateOnly'] = df['Deadline'].dt.date
    df['HasTime'] = (df['Deadline'].dt.hour != 0) | (df['Deadline'].dt.minute != 0)
    df['MatchedStringLength'] = df['MatchedString'].str.len()

    # Sort to prioritize for consolidation:
    # 1. Entries with a time component
    # 2. Entries with longer matched strings (more detail)
    df = df.sort_values(by=['Document', 'DateOnly', 'HasTime', 'MatchedStringLength'], ascending=[True, True, True, True])

    # Drop duplicates based on Document and DateOnly, keeping the last (most precise after sorting)
    df = df.drop_duplicates(subset=['Document', 'DateOnly'], keep='last')

    # Final sort by actual deadline
    df = df.sort_values("Deadline")
    df.reset_index(drop=True, inplace=True)

    # Clean up helper columns
    df = df.drop(columns=['DateOnly', 'HasTime', 'MatchedStringLength', 'MatchedString'])
else:
    print("No deadlines found to process.")

In [ ]:
deadline_list = df.to_dict("records")

deadline_list

[{'Document': 'Sample_Train_Ticket.pdf',
  'Deadline': Timestamp('2026-07-20 06:00:00'),
  'Event': 'Boarding Deadline: 20 July 2026, 06:00 AM'}]